# 🥉 第 2 阶段：手动工具调用（非 Agent）

## 🎯 阶段目标

让模型"间接"使用工具，但**由人类决定何时调用**。

> **工具调用 = 函数调用 + 结果注入**

这是学习 Agent 自动工具调用的重要铺垫。

---

## 🧠 核心概念

在这个阶段，**人类**负责决定：
- 是否需要调用工具
- 调用哪个工具
- 传递什么参数

然后将工具结果喂给模型，由模型生成最终回答。

---

## 📦 1. 加载配置

In [1]:
import json
import requests
from pathlib import Path

# 加载配置
config_path = Path.cwd() / ".." / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]

print("✅ 配置加载成功")
print(f"   模型：{MODEL_CONFIG['name']}")
print(f"   地址：{MODEL_CONFIG['base_url']}")

✅ 配置加载成功
   模型：qwen3.5-122b
   地址：http://36.134.71.54:8000


---

## 🔧 2. 定义工具函数

In [2]:
# 工具 1：天气查询
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    print(f"🔧 调用工具: get_weather({city})")
    
    weather_database = {
        "北京": {"天气": "晴天", "温度": "25°C", "湿度": "45%"},
        "上海": {"天气": "多云", "温度": "23°C", "湿度": "60%"},
        "广州": {"天气": "小雨", "温度": "28°C", "湿度": "80%"},
        "深圳": {"天气": "阴天", "温度": "27°C", "湿度": "75%"},
        "杭州": {"天气": "晴转多云", "温度": "24°C", "湿度": "55%"}
    }
    
    if city in weather_database:
        data = weather_database[city]
        return f"{city}的天气：{data['天气']}，温度{data['温度']}，湿度{data['湿度']}"
    else:
        return f"抱歉，暂不支持查询{city}的天气"

# 工具 2：计算器
def calculate(expression: str) -> str:
    """简单计算器"""
    print(f"🔧 调用工具: calculate({expression})")
    
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"计算结果：{expression} = {result}"
        else:
            return "表达式包含非法字符"
    except Exception as e:
        return f"计算错误：{str(e)}"

# 工具 3：获取当前时间
def get_time() -> str:
    """获取当前时间"""
    print(f"🔧 调用工具: get_time()")
    
    from datetime import datetime
    now = datetime.now()
    return f"当前时间：{now.strftime('%Y年%m月%d日 %H:%M:%S')}"

print("✅ 工具函数定义完成")
print("   - get_weather(city)")
print("   - calculate(expression)")
print("   - get_time()")

✅ 工具函数定义完成
   - get_weather(city)
   - calculate(expression)
   - get_time()


---

## 🚀 3. 定义 LLM 调用函数

In [3]:
def call_llm(messages: list) -> str:
    """调用 LLM 模型"""
    url = f"{MODEL_CONFIG['base_url']}/v1/chat/completions"
    
    payload = {
        "model": MODEL_CONFIG['name'],
        "messages": messages,
        "temperature": MODEL_CONFIG.get('temperature', 0.7),
        "max_tokens": MODEL_CONFIG.get('max_tokens', 100)
    }
    
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]
    except Exception as e:
        return f"❌ 错误：{str(e)}"

print("✅ LLM 调用函数定义完成")

✅ LLM 调用函数定义完成


---

## 🎮 4. 演示：手动工具调用

In [4]:
# 场景 1：天气查询
print("\n=== 场景 1：天气查询 ===")
user_question = "北京今天天气怎么样？"
print(f"👤 用户：{user_question}")

# 人类判断：需要调用天气工具
tool_result = get_weather("北京")
print(f"📊 工具结果：{tool_result}")

# 将工具结果喂给模型
messages = [
    {"role": "system", "content": "你是一个有帮助的助手"},
    {"role": "user", "content": f"根据以下信息回答问题：\n{tool_result}\n\n问题：{user_question}"}
]

answer = call_llm(messages)
print(f"🤖 助手：{answer}")


=== 场景 1：天气查询 ===
👤 用户：北京今天天气怎么样？
🔧 调用工具: get_weather(北京)
📊 工具结果：北京的天气：晴天，温度25°C，湿度45%
🤖 助手：Thinking Process:

1.  **Analyze the Request:**
    *   Input information: Beijing's weather (Sunny, 25°C, 45% humidity).
    *   Question: How is the weather in Beijing today? (北京今天天气怎么样？)
    *   Role: Helpful assistant.

2.  **Extract Key Information:**
    *   Location: Beijing (北京)
    *   Condition: Sunny (晴天)
    *   Temperature: 25°C
    *   Humidity: 45%

3.  **Formulate the Answer:**
    *   The answer should directly address the question using the provided information.
    *   Language: Chinese (matching the input and question).
    *   Tone: Helpful, clear, concise.

4.  **Drafting the Response:**
    *   Option 1 (Simple): 北京今天晴天，温度25°C，湿度45%。
    *   Option 2 (More natural): 北京今天的天气是晴天，气温25摄氏度，湿度为45%。
    *   Option 3 (Polite): 根据您提供的信息，北京今天天气晴朗，温度为25°C，湿度45%。

5.  **Selecting the Best Option:**
    *   Option 2 is natural and complete.
    *   Let's make it slightly conversationa

In [ ]:
# 场景 2：数学计算
print("\n=== 场景 2：数学计算 ===")
user_question = "123 + 456 等于多少？"
print(f"👤 用户：{user_question}")

# 人类判断：需要调用计算器工具
tool_result = calculate("123 + 456")
print(f"📊 工具结果：{tool_result}")

# 将工具结果喂给模型
messages = [
    {"role": "user", "content": f"根据以下计算结果回答：\n{tool_result}\n\n问题：{user_question}"}
]

answer = call_llm(messages)
print(f"🤖 助手：{answer}")

In [ ]:
# 场景 3：获取时间
print("\n=== 场景 3：获取时间 ===")
user_question = "现在几点了？"
print(f"👤 用户：{user_question}")

# 人类判断：需要调用时间工具
tool_result = get_time()
print(f"📊 工具结果：{tool_result}")

# 将工具结果喂给模型
messages = [
    {"role": "user", "content": f"根据以下信息回答：\n{tool_result}\n\n问题：{user_question}"}
]

answer = call_llm(messages)
print(f"🤖 助手：{answer}")

---

## 🚨 重要提示

> **这还不是真正的 Agent！**

在这个阶段，**人类**负责：
- 判断是否需要调用工具
- 选择调用哪个工具
- 提取工具参数

真正的 Agent 应该**自动**完成这些决策！

---

## ✅ 本阶段总结

你已经完成了：

1. ✅ 理解了工具的概念（Tool = Python 函数）
2. ✅ 学会了手动调用工具的流程
3. ✅ 掌握了将工具结果喂给模型的方法

---

## 🔗 下一步

进入 **[第 3 阶段：结构化输出](../step3/step3.ipynb)**，学习如何让模型输出可解析的结构。